# Project 1 — Bounded Asynchronous Crawler Simulator

Create a simulated web crawler using asyncio TaskGroup, bounded concurrency, deterministic fake pages, polite scheduling, deduplication, and failure reporting. No external network is contacted.

**Level:** Advanced portfolio · **Suggested time:** 4–8 hours · **Requirements:** Python 3.11+, Jupyter.

## Deliverables

- Define an explicit page map and allowed URLs
- Crawl discovered links without revisiting them
- Limit simultaneous fetches and bound total pages
- Capture failures and cancellation behavior
- Test success, missing pages and cycles

**Operating rules:** All demonstrations are offline, deterministic, and side-effect-limited. Read the full requirements and design before changing code. For a portfolio submission, move reusable logic to Python modules and add tests and a short architecture decision record.

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "projects"))
    sys.path.insert(0, os.getcwd())

## Step 1 — Offline page graph and provider

The fake provider models network responses by returning structured page records. A real crawler must obey robots policies, site terms, network limits and authorization; this project never sends real requests.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Page:
    url: str
    links: tuple[str, ...]
    text: str

PAGES = {
    "/": Page("/", ("/about", "/articles"), "home page"),
    "/about": Page("/about", ("/team",), "about page"),
    "/articles": Page("/articles", ("/team", "/missing"), "latest articles"),
    "/team": Page("/team", ("/",), "team page"),
}

def get_page(url):
    if not url.startswith("/"):
        raise ValueError("only local demo paths allowed")
    if url not in PAGES:
        raise FileNotFoundError(url)
    return PAGES[url]

print(get_page("/"))

**Review question:** Why use an allowlist of paths rather than making arbitrary outgoing requests?

## Step 2 — Bounded asynchronous fetch simulation

A semaphore limits simultaneous fetch tasks; an artificial asyncio.sleep(0) provides a suspension point without wasting time. The provider remains injected so tests can replace it.

In [ ]:
import sys; sys.path.insert(0, "../notebooks")
from lesson import run  # like asyncio.run(), but also works inside Jupyter and Colab
import asyncio

async def fetch_page(url, provider, semaphore, active):
    async with semaphore:
        active["count"] += 1
        active["peak"] = max(active["peak"], active["count"])
        try:
            await asyncio.sleep(0)
            return provider(url)
        finally:
            active["count"] -= 1

async def demo_fetch():
    semaphore = asyncio.Semaphore(2)
    active = {"count": 0, "peak": 0}
    pages = await asyncio.gather(
        *(fetch_page(path, get_page, semaphore, active) for path in ("/", "/about", "/team"))
    )
    return [page.url for page in pages], active["peak"]

print(run(demo_fetch()))

**Review question:** Does a semaphore enforce a maximum number of requests per second? Why not?

## Step 3 — Crawl by bounded frontiers

A breadth-first crawl maintains a pending frontier and visited set. Each frontier is processed under one TaskGroup so children have a known lifetime. Missing pages are recorded as errors without discarding successful pages. A max_pages bound prevents runaway traversal.

In [ ]:
import sys; sys.path.insert(0, "../notebooks")
from lesson import run  # like asyncio.run(), but also works inside Jupyter and Colab
import asyncio

async def crawl(start, provider=get_page, limit=2, max_pages=20):
    if limit < 1 or max_pages < 1:
        raise ValueError("positive limits required")
    visited, frontier = set(), [start]
    pages, errors = {}, {}
    semaphore = asyncio.Semaphore(limit)
    active = {"count": 0, "peak": 0}
    while frontier and len(visited) < max_pages:
        batch = []
        for url in frontier:
            if url not in visited and len(visited) + len(batch) < max_pages:
                batch.append(url)
        if not batch:
            break
        visited.update(batch)
        frontier = []
        async def safe_fetch(url):
            try:
                return url, await fetch_page(url, provider, semaphore, active), None
            except (FileNotFoundError, ValueError) as error:
                return url, None, str(error)
        async with asyncio.TaskGroup() as group:
            tasks = [group.create_task(safe_fetch(url)) for url in batch]
        for task in tasks:
            url, page, error = task.result()
            if error is not None:
                errors[url] = error
            else:
                pages[url] = page
                frontier.extend(link for link in page.links if link not in visited)
    return pages, errors, active["peak"]

pages, errors, peak = run(crawl("/"))
print(sorted(pages), sorted(errors), peak)
assert peak <= 2 and "/" in pages and "/missing" in errors

**Review question:** What does this implementation do if a page contains the same link multiple times?

## Step 4 — Summaries and deduplication tests

A crawler is only useful if its results are reproducible and duplicate URLs are not fetched repeatedly. Test cycles and missing pages with the injected fake provider; no actual domain or web service is touched.

In [ ]:
import sys; sys.path.insert(0, "../notebooks")
from lesson import run  # like asyncio.run(), but also works inside Jupyter and Colab
import unittest

class CrawlTests(unittest.TestCase):
    def test_cycle_stops(self):
        pages, errors, peak = run(crawl("/", max_pages=20))
        self.assertEqual(set(pages), {"/", "/about", "/articles", "/team"})
        self.assertLessEqual(peak, 2)
        self.assertIn("/missing", errors)
    def test_page_limit(self):
        pages, errors, peak = run(crawl("/", max_pages=1))
        self.assertEqual(set(pages), {"/"})
    def test_bad_limit(self):
        with self.assertRaises(ValueError):
            run(crawl("/", limit=0))

result = unittest.TextTestRunner(verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(CrawlTests))
assert result.wasSuccessful()

**Review question:** How would tests assert correct cancellation and cleanup if one task raises unexpectedly?

## Step 5 — Politeness, errors, and production boundaries

Real crawlers need host allowlists, robots handling, timeouts, bounded request rates, user-agent identification, and explicit consent where required. A concurrency semaphore alone does not implement host-level rate limiting. Keep this project as a simulation until such safeguards are designed.

In [ ]:
policy = {
    "max_pages": 20,
    "max_concurrency": 2,
    "live_network_enabled": False,
    "allowed_paths_only": True,
}
print(policy)
assert not policy["live_network_enabled"]

**Review question:** How would you add a per-host requests-per-second limiter without defeating structured concurrency?

## Independent extensions

Choose at least two and implement with tests. Describe error behavior and any scaling limits.

**Extension 1.** Add a simulated per-host rate limiter using an injectable clock and sleep.

In [ ]:
# TODO: Extension 1.
# Add a reproducible normal and failure case.

**Extension 2.** Support a maximum crawl depth and track parent URL.

In [ ]:
# TODO: Extension 2.
# Add a reproducible normal and failure case.

**Extension 3.** Demonstrate TaskGroup failure propagation and safe cancellation.

In [ ]:
# TODO: Extension 3.
# Add a reproducible normal and failure case.

**Extension 4.** Export a sorted CSV report of visited pages and failures.

In [ ]:
# TODO: Extension 4.
# Add a reproducible normal and failure case.

## Capstone rubric

- [ ] Requirements and public contracts documented.
- [ ] No leaked resources, orphaned tasks, or unbounded loops.
- [ ] Malformed inputs and failure paths covered by tests.
- [ ] No live network dependency or exposed secrets in test code.
- [ ] Rerunnable notebook without hidden state.
- [ ] Clear architecture explanation and trade-off analysis.

**Final:** Complete the Advanced assessment and prepare a project README highlighting your test evidence.